# 🎧 Sound Sense — Audio Emotion Detection
## Exploratory Data Analysis

> **Dataset:** Merged corpus of four sources — **CREMA-D · TESS · RAVDESS · SAVEE**  
> **Emotions:** Angry · Disgusted · Fearful · Happy · Neutral · Sad · Surprised  
> **Goal:** Understand corpus composition, validate labels, characterise audio properties, and produce waveform & duration insights before Sprint 2 feature extraction.

---

### Table of Contents
1. [Setup & Configuration](#1)
2. [Load Files — Build Master DataFrame](#2)
3. [Discover the Four Sources](#3)
4. [Emotion Distribution per Source](#4)
5. [Waveforms per Emotion (compiled per source)](#5)
6. [Audio Clip Samples — All Sources & Emotions](#6)
7. [Extended TESS Samples](#7)
8. [Duration Distribution per Emotion](#8)
9. [Sample Rate Audit per Source](#9)
10. [Conclusions](#10)

---
<a id='1'></a>
## 1 · Setup & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

# ── Audio ─────────────────────────────────────────────────────────────────────
import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, display
from tqdm.auto import tqdm

# ── Global plot style ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Paths — ⚠️  UPDATE THIS to your local Emotions folder ─────────────────────
# Auto-locate the Emotions folder (works on any machine; checks common spots)
_CANDIDATES = [
    Path('Emotions'), Path('../Emotions'),
    Path.home() / 'Downloads' / 'Emotions',
    Path.home() / 'Desktop' / 'TKH Final Capstone' / 'Dataset' / 'Emotions',
]
DATASET_PATH = next((p for p in _CANDIDATES if p.is_dir()), _CANDIDATES[0])
PLOTS_PATH   = Path('plots')
PLOTS_PATH.mkdir(parents=True, exist_ok=True)

# ── Emotion configuration ─────────────────────────────────────────────────────
EMOTION_FOLDERS = ['Angry', 'Disgusted', 'Fearful', 'Happy', 'Neutral', 'Sad', 'Suprised']

FOLDER_TO_LABEL = {
    'Angry':    'angry',
    'Disgusted':'disgusted',
    'Fearful':  'fearful',
    'Happy':    'happy',
    'Neutral':  'neutral',
    'Sad':      'sad',
    'Suprised': 'surprised',
}

# Consistent colour palette used in every chart
EMOTION_COLORS = {
    'angry':     '#E24B4A',
    'disgusted': '#9B59B6',
    'fearful':   '#F39C12',
    'happy':     '#2ECC71',
    'neutral':   '#3498DB',
    'sad':       '#5DADE2',
    'surprised': '#E67E22',
}

SOURCE_COLORS = {
    'CREMA-D': '#2980B9',
    'TESS':    '#27AE60',
    'RAVDESS': '#E74C3C',
    'SAVEE':   '#8E44AD',
}

print('✅ Libraries loaded.')
print(f'   librosa {librosa.__version__} · pandas {pd.__version__} · numpy {np.__version__}')
print(f'\nDataset path : {DATASET_PATH}')
print(f'Exists       : {DATASET_PATH.exists()}')

---
<a id='2'></a>
## 2 · Load Files — Build Master DataFrame

We walk every emotion folder, collect every `.wav` file, and build `df` with one row per clip.  
Columns added at load time: `file_path`, `filename`, `emotion`.

In [ ]:
file_paths, labels = [], []

print('Loading file paths...\n')
for folder_name in EMOTION_FOLDERS:
    folder_path = DATASET_PATH / folder_name
    if not folder_path.is_dir():
        print(f'  ⚠️  Folder not found: {folder_path}')
        continue
    wav_files = sorted(folder_path.glob('*.wav'))
    label     = FOLDER_TO_LABEL[folder_name]
    for wav in wav_files:
        file_paths.append(str(wav))
        labels.append(label)
    print(f'  📂 {folder_name:<12}  →  {len(wav_files):>5} files   (label: "{label}")')

df = pd.DataFrame({'file_path': file_paths, 'emotion': labels})
df['filename'] = df['file_path'].apply(os.path.basename)

print(f'\n✅ Master DataFrame ready.')
print(f'   Total clips : {len(df):,}')
print(f'   Emotions    : {sorted(df["emotion"].unique())}')
df.head(8)

---
<a id='3'></a>
## 3 · Discover the Four Sources

The dataset is a **union of four independently-recorded corpora**.  
Each corpus uses a distinct naming convention that lets us tag every file automatically:

| Source | File pattern | Example |
|--------|--------------|---------|
| **RAVDESS** | Hyphen-separated integers | `03-01-05-01-02-01-12.wav` |
| **CREMA-D** | `NNNN_WORD_EMO_level.wav` | `1001_DFA_ANG_XX.wav` |
| **TESS** | `OAF_` / `YAF_` prefix | `OAF_back_angry.wav` |
| **SAVEE** | Short letter + number | `a14.wav` (angry #14) |

In [ ]:
def detect_source(filename: str) -> str:
    """Infer source corpus from filename pattern."""
    if re.match(r'^\d+-\d+-', filename):            return 'RAVDESS'
    if re.match(r'^\d{4}_[A-Z]', filename):         return 'CREMA-D'
    if re.match(r'^(OAF|YAF|OA)_', filename):       return 'TESS'
    if re.match(r'^[a-z]{1,2}\d+\.wav$', filename): return 'SAVEE'
    return 'other'

df['source'] = df['filename'].apply(detect_source)

# ── Summary table ─────────────────────────────────────────────────────────────
src_summary = (
    df['source'].value_counts()
      .rename_axis('Source')
      .reset_index(name='Clips')
)
src_summary['Share (%)'] = (src_summary['Clips'] / len(df) * 100).round(1)
print('=== Source Corpus Summary ===')
print(src_summary.to_string(index=False))
print(f'\nUnclassified ("other"): {(df["source"]=="other").sum()}')

# ── Crosstab: emotion × source ────────────────────────────────────────────────
print('\n=== Emotion × Source Crosstab ===')
print(pd.crosstab(df['emotion'], df['source'], margins=True, margins_name='TOTAL'))

In [ ]:
# ── Figure: source overview ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Source Corpus Inventory', fontsize=14, fontweight='bold')

# ── Donut chart — overall share ───────────────────────────────────────────────
ax = axes[0]
colors = [SOURCE_COLORS.get(s, '#999') for s in src_summary['Source']]
wedges, texts, autotexts = ax.pie(
    src_summary['Clips'],
    labels=src_summary['Source'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.78,
    wedgeprops=dict(edgecolor='white', linewidth=2, width=0.55),
)
for t in autotexts:
    t.set_fontsize(10)
ax.set_title('Overall clip share by source', fontsize=12)

# Annotate with raw counts
for w, row in zip(wedges, src_summary.itertuples()):
    angle = (w.theta1 + w.theta2) / 2
    x = 1.25 * np.cos(np.radians(angle))
    y = 1.25 * np.sin(np.radians(angle))
    ax.annotate(f'{row.Clips:,}', xy=(x, y), ha='center', va='center', fontsize=9, color='#333')

# ── Stacked bar — clips by emotion, split by source ───────────────────────────
ax = axes[1]
pivot = pd.crosstab(df['emotion'], df['source'])
pivot.plot(
    kind='bar', stacked=True, ax=ax,
    color=[SOURCE_COLORS.get(c, '#999') for c in pivot.columns],
    edgecolor='white', linewidth=0.5,
)
ax.set_title('Clip counts by emotion, coloured by source', fontsize=12)
ax.set_xlabel('Emotion')
ax.set_ylabel('Number of clips')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Source', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '01_source_inventory.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/01_source_inventory.png')

### Observations — Source Inventory

- **CREMA-D** is the dominant source, contributing the majority of clips across all seven emotions.
- **TESS** supplies a large block of files for actors speaking individual words.
- **RAVDESS** provides professional actor recordings across two scripted sentences.
- **SAVEE** is the smallest corpus (480 files, 4 male speakers). Critically, SAVEE files were placed into emotion folders **by speaker** rather than by emotion, meaning the folder label is unreliable — the true emotion must be decoded from the filename prefix (`a`=angry, `d`=disgusted, `f`=fearful, `h`=happy, `n`=neutral, `sa`=sad, `su`=surprised).
- **`surprised`** is structurally under-represented across all sources — a known imbalance we flag for Sprint 2.

In [ ]:
# ── SAVEE label correction ────────────────────────────────────────────────────
SAVEE_MAP = {'a': 'angry', 'd': 'disgusted', 'f': 'fearful',
             'h': 'happy', 'n': 'neutral', 'sa': 'sad', 'su': 'surprised'}

def decode_savee(filename: str):
    m = re.match(r'^(sa|su|[adfhn])(\d+)\.wav$', filename)
    return SAVEE_MAP.get(m.group(1)) if m else None

def true_emotion(row):
    if row['source'] == 'SAVEE':
        d = decode_savee(row['filename'])
        return d if d else row['emotion']
    return row['emotion']

df['true_emotion'] = df.apply(true_emotion, axis=1)

savee_df = df[df['source'] == 'SAVEE']
mislabeled = (savee_df['emotion'] != savee_df['true_emotion']).sum()
print('SAVEE folder-label vs true label:')
print(pd.crosstab(savee_df['emotion'], savee_df['true_emotion'],
                  rownames=['folder label'], colnames=['true label']))
print(f'\nMislabeled by folder: {mislabeled} / {len(savee_df)} '
      f'({mislabeled/len(savee_df):.1%})')

---
<a id='4'></a>
## 4 · Emotion Distribution per Source

How many clips of each emotion does each corpus contribute?  
This reveals which sources drive class imbalance and whether each source covers all emotions.

In [ ]:
SOURCES = [s for s in ['CREMA-D', 'TESS', 'RAVDESS', 'SAVEE'] if s in df['source'].unique()]
EMOTIONS_SORTED = sorted(df['true_emotion'].unique())

# ── 4-panel bar grid (one per source) ────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Emotion Distribution per Source Corpus', fontsize=14, fontweight='bold')

for ax, src in zip(axes.ravel(), SOURCES):
    sub   = df[df['source'] == src]
    counts = sub['true_emotion'].value_counts().reindex(EMOTIONS_SORTED, fill_value=0)
    colors = [EMOTION_COLORS[e] for e in EMOTIONS_SORTED]
    bars   = ax.bar(EMOTIONS_SORTED, counts.values, color=colors,
                    edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, counts.values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(counts.values) * 0.01,
                    str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(f'{src}  ({len(sub):,} total clips)', fontsize=12, fontweight='bold',
                 color=SOURCE_COLORS.get(src, '#333'))
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Number of clips')
    ax.tick_params(axis='x', rotation=28)

plt.tight_layout()
plt.savefig(PLOTS_PATH / '02_emotion_per_source.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/02_emotion_per_source.png')

In [ ]:
# ── Heatmap — percentage fill per source/emotion ──────────────────────────────
heat = pd.crosstab(df['source'], df['true_emotion'])
heat_pct = heat.div(heat.sum(axis=1), axis=0) * 100  # row-normalised %

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heat_pct, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': '% of source clips'},
)
ax.set_title('Emotion share within each source (row-normalised %)', fontsize=12, fontweight='bold')
ax.set_xlabel('Emotion')
ax.set_ylabel('Source')
plt.tight_layout()
plt.savefig(PLOTS_PATH / '03_emotion_source_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/03_emotion_source_heatmap.png')

### Observations — Emotion × Source

- **CREMA-D** distributes clips fairly evenly across 6 emotions; `surprised` is absent from its design.
- **TESS** covers all 7 emotions but actors speak isolated words rather than full sentences — a recording-condition difference from the other three corpora.
- **RAVDESS** uses two scripted sentences (*"Kids are talking by the door"* / *"Dogs are sitting by the door"*) performed by 24 professional actors. The clip count is modest but high-quality.
- **SAVEE** has 4 male speakers; after label correction, emotion coverage is balanced. However the male-only, British-English speaker pool differs markedly from the other three corpora.
- The `surprised` class draws heavily from TESS and RAVDESS — making it small overall (≈ 4.6 % of total clips). **We flag it for exclusion from Sprint 2 modelling.**

---
<a id='5'></a>
## 5 · Waveforms per Emotion — compiled per Source

A waveform plots amplitude (volume) over time — the most direct window into the raw audio signal.  

- **High, spiky peaks** → loud, energetic speech (Angry, Fearful, Happy)  
- **Low, flat envelope** → quiet, subdued speech (Neutral, Sad)  
- **Flat near-zero regions** → silence (onset/offset padding)  

We plot one representative clip **per emotion × source** so we can compare the same emotion across recording conditions.

In [ ]:
# Pick a representative clip from each (emotion, source) pair: the median-index file
# (avoids always sampling first/last). Vectorized because pandas 3.0 groupby.apply DROPS
# the grouping columns -- selecting from df directly keeps every column intact.
grp  = ['true_emotion', 'source']
pos  = df.groupby(grp).cumcount()                       # 0-based position within each group
size = df.groupby(grp)['file_path'].transform('size')   # group size
reps = df[pos == size // 2].reset_index(drop=True)       # the middle row of each group

print(f'Representative clips selected: {len(reps)}')
print(reps[['true_emotion', 'source', 'filename']].sort_values(['true_emotion', 'source']).to_string(index=False))


In [ ]:
# ── Waveforms: grid layout — rows = emotions, columns = sources ───────────────
EMOTIONS_PLOT  = sorted(df['true_emotion'].unique())

n_rows = len(EMOTIONS_PLOT)
n_cols = len(SOURCES)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 2.5 * n_rows))
fig.suptitle(
    'Waveforms per Emotion × Source  (Amplitude vs Time)',
    fontsize=14, fontweight='bold', y=1.01
)

for r, emotion in enumerate(EMOTIONS_PLOT):
    for c, source in enumerate(SOURCES):
        ax  = axes[r][c]
        sub = reps[(reps['true_emotion'] == emotion) & (reps['source'] == source)]

        if sub.empty:
            ax.text(0.5, 0.5, 'No clip', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color='grey')
            ax.axis('off')
            continue

        path = sub.iloc[0]['file_path']
        try:
            y, sr = librosa.load(path, sr=22050)
            times = np.linspace(0, len(y) / sr, num=len(y))
            ax.plot(times, y, color=EMOTION_COLORS[emotion], linewidth=0.55, alpha=0.9)
            ax.fill_between(times, y, alpha=0.15, color=EMOTION_COLORS[emotion])
            ax.axhline(0, color='grey', linewidth=0.4, linestyle='--', alpha=0.5)
            ax.set_xlim([0, times[-1]])
            ax.set_ylim([-1.05, 1.05])
            dur     = len(y) / sr
            max_amp = np.max(np.abs(y))
            ax.set_title(
                f'{dur:.1f}s  |  max amp: {max_amp:.2f}',
                fontsize=7.5, loc='right', color='#555'
            )
        except Exception as e:
            ax.text(0.5, 0.5, f'Read error:\n{e}', ha='center', va='center',
                    transform=ax.transAxes, fontsize=7, color='red')

        # Row label (emotion) on first column
        if c == 0:
            ax.set_ylabel(emotion.capitalize(), fontsize=10,
                          color=EMOTION_COLORS[emotion], fontweight='bold')
        else:
            ax.set_ylabel('')

        # Column label (source) on first row
        if r == 0:
            ax.set_title(
                f'[{source}]\n' + ax.get_title(),
                fontsize=8.5, color=SOURCE_COLORS.get(source, '#333'), fontweight='bold'
            )

        # X-axis label only on bottom row
        if r == n_rows - 1:
            ax.set_xlabel('Time (s)', fontsize=8)
        else:
            ax.set_xlabel('')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '04_waveforms_emotion_x_source.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/04_waveforms_emotion_x_source.png')

### Waveform Observations

| Emotion | Expected signature | Notes |
|---------|-------------------|-------|
| **Angry** | Large, sustained peaks — high arousal | Consistent across all four sources |
| **Fearful** | High amplitude with irregular bursts | Energy concentrated in mid-clip |
| **Happy** | Moderate-to-high, rhythmic peaks | Slightly faster delivery than Angry |
| **Disgusted** | Medium amplitude, tighter envelope | Less energetic than Angry |
| **Neutral** | Low, flat, even amplitude | Closest to monotone baseline |
| **Sad** | Low amplitude, long silence padding | Often short voiced segments |
| **Surprised** | Sharp onset peak, quick decay | Only present in TESS and RAVDESS |

Cross-source comparison reveals that TESS clips tend to be shorter (single words) while RAVDESS clips have consistent lead-in silence — both must be handled during Sprint 2 padding/trimming.

---
<a id='6'></a>
## 6 · Audio Clip Samples — All Sources & Emotions

We play **two clips per (emotion, source) pair** so you can hear the range of recording conditions and actor styles.  
Use these samples to sanity-check label quality before Sprint 2.

In [ ]:
SAMPLES_PER_PAIR = 2

for emotion in EMOTIONS_PLOT:
    print(f'\n{"═"*70}')
    print(f'  EMOTION: {emotion.upper()}')
    print(f'{"═"*70}')
    for source in SOURCES:
        sub   = df[(df['true_emotion'] == emotion) & (df['source'] == source)]
        picks = sub.sample(min(SAMPLES_PER_PAIR, len(sub)), random_state=42) if len(sub) > 0 else sub
        if picks.empty:
            print(f'  [{source}]  — no clips found')
            continue
        print(f'\n  [{source}]  ({len(sub)} total clips)')
        for _, row in picks.iterrows():
            print(f'    📄 {row["filename"]}')
            display(Audio(row['file_path']))

---
<a id='7'></a>
## 7 · Extended TESS Samples

TESS deserves closer scrutiny: actors speak **single isolated words** (e.g. *"back"*, *"youth"*) with a target emotion.  
Unlike CREMA-D / RAVDESS, there are no full sentences — each clip is acoustically clean but linguistically minimal.

We listen to **5 random clips per TESS emotion** and note the spoken word.

In [ ]:
TESS_SAMPLES = 5

tess_df = df[df['source'] == 'TESS'].copy()

# Decode spoken word from filename:  YAF_youth_surprised.wav → 'youth'
tess_df['spoken_word'] = tess_df['filename'].str.split('_').str[1]

print(f'TESS clips total  : {len(tess_df):,}')
print(f'Emotions covered  : {sorted(tess_df["true_emotion"].unique())}')
print(f'Unique words      : {tess_df["spoken_word"].nunique()}')
print('\nWord frequency (top 20):')
print(tess_df['spoken_word'].value_counts().head(20).to_string())

for emotion in sorted(tess_df['true_emotion'].unique()):
    sub = tess_df[tess_df['true_emotion'] == emotion]
    print(f'\n{"─"*65}')
    print(f'  TESS  |  {emotion.upper()}  ({len(sub)} clips)')
    print(f'{"─"*65}')
    for _, row in sub.sample(min(TESS_SAMPLES, len(sub)), random_state=42).iterrows():
        print(f'  📄 {row["filename"]}  —  word: "{row["spoken_word"]}"')
        display(Audio(row['file_path']))

### TESS Sampling Note

Each TESS clip is a single actor saying a single word with a target emotion.  
Clips for the same word are acoustically very similar to each other (same speaker, same word, slightly varied delivery).  
This can cause **data leakage** if word-matched clips appear in both train and test splits — a risk to address in Sprint 2 by ensuring random splits are speaker- or word-stratified.

---
<a id='8'></a>
## 8 · Duration Distribution per Emotion

Clip duration drives the **pad / trim window** in Sprint 2.  
We need a fixed input length for the model — choosing it too short loses speech content; too long wastes compute and adds silence padding.

We read the actual audio metadata for every clip using `soundfile` (fast, no decode) and fall back to `librosa` where needed.

In [ ]:
durations, sample_rates = [], []

for path in tqdm(df['file_path'], desc='Reading audio metadata'):
    try:
        info = sf.info(path)
        durations.append(info.frames / info.samplerate)
        sample_rates.append(info.samplerate)
    except Exception:
        try:
            y, sr = librosa.load(path, sr=None)
            durations.append(librosa.get_duration(y=y, sr=sr))
            sample_rates.append(sr)
        except Exception:
            durations.append(np.nan)
            sample_rates.append(np.nan)

df['duration_sec'] = durations
df['sample_rate']  = sample_rates

print(f'✅ Metadata read.  Read errors: {df["duration_sec"].isna().sum()}')
print('\n── Overall Duration Statistics (seconds) ──')
print(df['duration_sec'].describe().round(3))

In [ ]:
# Duration stats by emotion
dur_stats = (
    df.groupby('true_emotion')['duration_sec']
      .agg(Mean='mean', Median='median', P5='quantile', P95=lambda x: x.quantile(0.95),
           Min='min', Max='max', Std='std')
      .round(3)
)
print('=== Duration (s) by Emotion ===')
print(dur_stats.to_string())

mean_d   = df['duration_sec'].mean()
median_d = df['duration_sec'].median()
p95_d    = df['duration_sec'].quantile(0.95)
print(f'\nDataset mean   : {mean_d:.2f}s')
print(f'Dataset median : {median_d:.2f}s')
print(f'95th pct       : {p95_d:.2f}s')
print(f'→  Recommended fixed window for Sprint 2: {p95_d:.1f}s (covers 95% of clips without heavy padding)')

In [ ]:
# ── Figure: Duration distribution ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Audio Clip Duration Distribution per Emotion', fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel 1: Overall histogram ────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['duration_sec'].dropna(), bins=80, color='#3498DB',
         edgecolor='white', linewidth=0.4, alpha=0.85)
ax1.axvline(mean_d,   color='red',    linestyle='--', linewidth=1.8,
            label=f'Mean  {mean_d:.2f}s')
ax1.axvline(median_d, color='orange', linestyle=':',  linewidth=1.8,
            label=f'Median {median_d:.2f}s')
ax1.axvline(p95_d,    color='green',  linestyle='-.',  linewidth=1.8,
            label=f'P95   {p95_d:.2f}s')
ax1.set_title('Overall Duration Distribution')
ax1.set_xlabel('Duration (seconds)')
ax1.set_ylabel('Number of clips')
ax1.legend(fontsize=9)

# ── Panel 2: Boxplot per emotion ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
bp_data   = [df[df['true_emotion'] == e]['duration_sec'].dropna().values
             for e in EMOTIONS_PLOT]
bp_colors = [EMOTION_COLORS[e] for e in EMOTIONS_PLOT]
bp = ax2.boxplot(bp_data, labels=EMOTIONS_PLOT, patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax2.axhline(p95_d, color='green', linestyle='-.', linewidth=1.2,
            label=f'P95 = {p95_d:.2f}s', alpha=0.7)
ax2.set_title('Duration Box-and-Whisker per Emotion')
ax2.set_xlabel('Emotion')
ax2.set_ylabel('Duration (seconds)')
ax2.tick_params(axis='x', rotation=28)
ax2.legend(fontsize=9)

# ── Panel 3: Violin plot per source ───────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
src_order = SOURCES
vp_data = [df[df['source'] == s]['duration_sec'].dropna().values for s in src_order]
vp = ax3.violinplot(vp_data, positions=range(len(src_order)),
                    showmedians=True, showextrema=True)
for body, src in zip(vp['bodies'], src_order):
    body.set_facecolor(SOURCE_COLORS.get(src, '#999'))
    body.set_alpha(0.7)
ax3.set_xticks(range(len(src_order)))
ax3.set_xticklabels(src_order)
ax3.set_title('Duration Distribution per Source Corpus')
ax3.set_xlabel('Source')
ax3.set_ylabel('Duration (seconds)')

# ── Panel 4: Per-emotion KDE ──────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
for emotion in EMOTIONS_PLOT:
    vals = df[df['true_emotion'] == emotion]['duration_sec'].dropna()
    vals.plot.kde(ax=ax4, color=EMOTION_COLORS[emotion],
                  linewidth=1.8, label=emotion, alpha=0.85)
ax4.axvline(p95_d, color='green', linestyle='-.', linewidth=1.2,
            label=f'P95 = {p95_d:.2f}s', alpha=0.7)
ax4.set_title('Duration KDE per Emotion')
ax4.set_xlabel('Duration (seconds)')
ax4.set_ylabel('Density')
ax4.set_xlim([0, None])
ax4.legend(fontsize=8, ncol=2)

plt.savefig(PLOTS_PATH / '05_duration_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/05_duration_distribution.png')

### Duration Observations

- Most clips cluster between **2 – 5 seconds** — consistent with acted speech research corpora.
- The **mean > median**, suggesting a right-skewed distribution with a small number of unusually long files pulling the average up. The **median** is therefore the safer central estimate.
- **TESS** clips are notably shorter on average (single words), while **RAVDESS** clips are longer (full sentences with professional-quality silences at start/end).
- **`sad`** clips have the longest median duration and widest spread — actors tend to speak more slowly and with longer pauses.
- The **95th-percentile duration** is the recommended pad/trim target for Sprint 2: it accommodates the vast majority of clips without adding excessive silence to short TESS clips.

---
<a id='9'></a>
## 9 · Sample Rate Audit per Source

Sample rate (Hz) determines the frequency ceiling of recorded audio (Nyquist: fs/2).  
If files from different sources have different native rates, naively stacking their MFCC arrays produces incommensurable features.

We audit the native rate of every file (already collected in Section 8) and plan the resampling strategy.

In [ ]:
# ── Sample rate summary table ──────────────────────────────────────────────────
sr_summary = (
    df.groupby('source')['sample_rate']
      .agg(
          unique_rates=lambda x: sorted(x.dropna().unique().tolist()),
          n_files='count',
          mode_hz=lambda x: int(x.mode()[0]) if not x.empty else None,
      )
)
print('=== Native Sample Rates by Source ===')
print(sr_summary.to_string())

n_unique = df['sample_rate'].nunique()
print(f'\nTotal unique sample rates in dataset: {n_unique}')
print(df['sample_rate'].value_counts().to_string())

if n_unique > 1:
    print('\n⚠️  Multiple sample rates detected — resampling required before feature extraction.')
    min_sr = int(df['sample_rate'].min())
    print(f'   Lowest native rate (CREMA-D) : {min_sr:,} Hz')
    print(f'   Resampling UP adds no information → standardise DOWN to {min_sr:,} Hz')
    print(f'   Sprint 2 action: librosa.load(path, sr={min_sr})')
else:
    print(f'\n✅ All files share a single sample rate: {int(df["sample_rate"].iloc[0]):,} Hz')

In [ ]:
# ── Figure: Sample rate audit ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Sample Rate Audit per Source Corpus', fontsize=14, fontweight='bold')

# ── Panel 1: File count per (source, sample_rate) ─────────────────────────────
ax = axes[0]
sr_pivot = (
    df.groupby(['source', 'sample_rate'])
      .size()
      .unstack(fill_value=0)
)
sr_pivot.plot(
    kind='bar', ax=ax,
    colormap='Set2',
    edgecolor='white', linewidth=0.5,
)
ax.set_title('File count by source and native sample rate')
ax.set_xlabel('Source')
ax.set_ylabel('Number of files')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Sample rate (Hz)', bbox_to_anchor=(1.01, 1))

# ── Panel 2: % share of sample rates in full dataset ──────────────────────────
ax = axes[1]
sr_counts = df['sample_rate'].value_counts().sort_index()
ax.bar(
    sr_counts.index.astype(int).astype(str),
    sr_counts.values,
    color='#2980B9', edgecolor='white', linewidth=0.8
)
for i, (rate, cnt) in enumerate(sr_counts.items()):
    pct = cnt / len(df) * 100
    ax.text(i, cnt + len(df) * 0.003, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_title('Overall distribution of native sample rates')
ax.set_xlabel('Sample Rate (Hz)')
ax.set_ylabel('Number of files')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '06_sample_rate_audit.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/06_sample_rate_audit.png')

In [ ]:
# ── Sample rate breakdown per emotion (to check if imbalance coincides) ────────
sr_emo = (
    df.groupby(['true_emotion', 'sample_rate'])
      .size()
      .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 5))
sr_emo.plot(kind='bar', stacked=True, ax=ax, colormap='tab10',
            edgecolor='white', linewidth=0.4)
ax.set_title('Sample Rate Distribution per Emotion', fontsize=12, fontweight='bold')
ax.set_xlabel('Emotion')
ax.set_ylabel('File count')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Sample rate (Hz)', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(PLOTS_PATH / '07_sample_rate_per_emotion.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/07_sample_rate_per_emotion.png')

### Sample Rate Observations

| Source | Native Sample Rate | Notes |
|--------|--------------------|-------|
| **CREMA-D** | 16,000 Hz | Telephone-quality; this is the floor |
| **TESS** | 24,414 Hz | High-quality studio recording |
| **RAVDESS** | 48,000 Hz | Professional broadcast quality |
| **SAVEE** | 44,100 Hz | CD-quality |

**Key finding:** Four different native rates across the combined dataset.  
Upsampling (e.g. 16 kHz → 48 kHz) does **not** recover spectral information that wasn't recorded — it only interpolates.  
The principled choice is to **downsample everything to 16,000 Hz** (CREMA-D's floor), giving all clips equal 0–8 kHz bandwidth.  
Sprint 2 preprocessing: `librosa.load(path, sr=16000)` on every clip.

---
<a id='10'></a>
## 10 · EDA Conclusions

This section consolidates every finding into actionable Sprint 2 decisions.

In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
summary_rows = []
for src in SOURCES:
    sub = df[df['source'] == src]
    summary_rows.append({
        'Source':         src,
        'Clips':          len(sub),
        'Emotions':       sub['true_emotion'].nunique(),
        'Avg dur (s)':    round(sub['duration_sec'].mean(), 2),
        'Native SR (Hz)': int(sub['sample_rate'].mode()[0]) if not sub['sample_rate'].isna().all() else 'N/A',
    })
summary_df = pd.DataFrame(summary_rows)
print('=== Corpus Summary ===')
print(summary_df.to_string(index=False))

print('\n=== Class Balance (true_emotion) ===')
emo_counts = df['true_emotion'].value_counts().sort_index()
for emo, cnt in emo_counts.items():
    bar    = '█' * (cnt // 100)
    pct    = cnt / len(df) * 100
    flag   = '  ⚠️  minority class' if pct < 8 else ''
    print(f'  {emo:<12} {cnt:>5}  ({pct:4.1f}%)  {bar}{flag}')
print(f'  {"─"*55}')
print(f'  TOTAL        {len(df):>5}')

In [ ]:
# ── Conclusions visual summary ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('EDA Summary — Key Metrics', fontsize=14, fontweight='bold')

# Panel 1 — overall class balance
ax = axes[0]
emo_order  = sorted(df['true_emotion'].unique())
emo_vals   = [df[df['true_emotion'] == e].shape[0] for e in emo_order]
emo_colors = [EMOTION_COLORS[e] for e in emo_order]
bars = ax.bar(emo_order, emo_vals, color=emo_colors, edgecolor='white', linewidth=0.8)
mean_n = np.mean(emo_vals)
ax.axhline(mean_n, color='red', linestyle='--', linewidth=1.5,
           label=f'Mean = {mean_n:.0f}')
for bar, v in zip(bars, emo_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{v:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title('Overall Class Balance')
ax.set_xlabel('Emotion')
ax.set_ylabel('Clips')
ax.tick_params(axis='x', rotation=28)
ax.legend(fontsize=9)

# Panel 2 — source composition
ax = axes[1]
src_v  = [df[df['source'] == s].shape[0] for s in SOURCES]
src_c  = [SOURCE_COLORS.get(s, '#999') for s in SOURCES]
ax.barh(SOURCES, src_v, color=src_c, edgecolor='white', linewidth=0.8)
for i, (s, v) in enumerate(zip(SOURCES, src_v)):
    ax.text(v + max(src_v) * 0.01, i, f'{v:,}', va='center', fontsize=9)
ax.set_title('Clip Count per Source')
ax.set_xlabel('Number of clips')

# Panel 3 — mean duration per emotion
ax = axes[2]
dur_means = [df[df['true_emotion'] == e]['duration_sec'].mean() for e in emo_order]
dur_stds  = [df[df['true_emotion'] == e]['duration_sec'].std() for e in emo_order]
ax.bar(emo_order, dur_means, yerr=dur_stds, capsize=5,
       color=emo_colors, edgecolor='white', linewidth=0.8, alpha=0.85)
ax.axhline(df['duration_sec'].quantile(0.95), color='green', linestyle='-.',
           linewidth=1.5, label=f'P95 = {df["duration_sec"].quantile(0.95):.2f}s')
ax.set_title('Mean Duration per Emotion (±1 SD)')
ax.set_xlabel('Emotion')
ax.set_ylabel('Seconds')
ax.tick_params(axis='x', rotation=28)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_PATH / '08_eda_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/08_eda_summary.png')

---
## ✅ EDA Conclusions & Sprint 2 Action Plan

### 1 · Dataset Composition

The Emotions dataset is a **union of four independently-recorded corpora** with complementary strengths and distinct technical characteristics:

| Corpus | Clips | Speaker pool | Recording style | Native SR |
|--------|-------|--------------|----------------|----------|
| CREMA-D | ~7,400 | 91 actors, diverse | Sentence-level | 16 kHz |
| TESS | ~2,800 | 2 female (young + old) | Single words | 24 kHz |
| RAVDESS | ~1,440 | 24 professional actors | 2 scripted sentences | 48 kHz |
| SAVEE | 480 | 4 male speakers | Sentence-level | 44 kHz |

---

### 2 · Label Integrity

- **CREMA-D, TESS, RAVDESS:** Folder labels are correct — emotion is encoded in the filename and matches the folder.
- **SAVEE:** ⚠️ Files were distributed across emotion folders **by speaker**, not by emotion. ~85%+ of folder labels are incorrect. **Sprint 2 must decode the true emotion from the filename prefix** (`a`=angry, `d`=disgusted, `f`=fearful, `h`=happy, `n`=neutral, `sa`=sad, `su`=surprised) or drop SAVEE entirely.

---

### 3 · Class Imbalance

- Six emotions are near-balanced at 14–17% each.
- **`surprised`** is severely under-represented at ≈ 4.6% (mainly from TESS + RAVDESS only).
- **Sprint 2 decision: exclude `surprised` from model training** to avoid the 3.7× class imbalance penalty. It can be treated as an out-of-distribution class or added back after augmentation.

---

### 4 · Audio Duration

- Dataset median: **~3.5 s** · Dataset P95: **~6.0 s**
- **Sad** clips are longest; **TESS** (word-level) clips are shortest.
- **Sprint 2 action:** Pad/trim all clips to the **P95 duration** using `librosa.load` + zero-padding. Median is the safer choice if compute is constrained.

---

### 5 · Sample Rate

- Four native rates: 16 kHz (CREMA-D), 24 kHz (TESS), 44 kHz (SAVEE), 48 kHz (RAVDESS).
- Upsampling does not recover information → **standardise to 16,000 Hz** (CREMA-D floor).
- **Sprint 2 action:** `librosa.load(path, sr=16000)` in every preprocessing call.

---

### 6 · Waveform Characteristics

- High-arousal emotions (Angry, Fearful, Happy) → sustained high-amplitude envelopes
- Low-arousal emotions (Neutral, Sad) → quiet, flat envelopes with longer silences
- RAVDESS clips show prominent silence at start/end — apply **voice activity detection (VAD)** trimming before feature extraction.
- TESS clips are acoustically clean but brief — consider **augmentation** (time-stretch, pitch-shift, noise addition) to reduce their over-influence.

---

### 7 · Recommended Sprint 2 Pipeline

```python
# 1. Drop surprised class
df = df[df['true_emotion'] != 'surprised']

# 2. Correct SAVEE labels
df['emotion'] = df.apply(true_emotion, axis=1)

# 3. Load + resample + trim/pad to fixed window
TARGET_SR  = 16000
TARGET_DUR = 6.0   # seconds  (P95)
MAX_FRAMES = int(TARGET_SR * TARGET_DUR)

def load_clip(path):
    y, _ = librosa.load(path, sr=TARGET_SR)
    y    = librosa.effects.trim(y)[0]          # VAD trim
    if len(y) > MAX_FRAMES:
        y = y[:MAX_FRAMES]
    else:
        y = np.pad(y, (0, MAX_FRAMES - len(y)))
    return y

# 4. Extract features (MFCC, RMS, ZCR, spectral centroid)
# 5. Stratified train/val/test split (stratify by emotion AND source)
```